# Cinema ticket forecast — v6 (public LB 0.4777)

Predict daily `total_ticket` for D4–D10 of each (movie, cinema) pair from its D1–D3 sales.

**Metric:** MASE = `mean(|y - p| / scale)`, `scale = max((D1+D2+D3)/3, 1)` per pair.

**Pipeline**
1. **Wide-release alignment** — 58% of train titles first appear as a 1-cinema preview. Test D1 is the
   wide opening, so train D1 is re-anchored to the first day a title reaches ≥30% of its peak cinema count.
2. **Dense D1–D10 cohorts** — every pair active on D3 gets all 7 target days; days with no row (film pulled
   from the cinema) are 0, exactly like the test grid.
3. **Features** — D1–D3 shape, shows, occupancy, horizon, weekday, release weekday, holidays, city price tier,
   genre/age rating, cinema id, plus **screen competition**: other titles opening at the same cinema / nationally
   during D4–D10 (taken only from other titles' D1–D3 windows, available identically in `test_history.csv`).
4. **Model** — target `ticket / scale` with L1 loss (so training MAE == MASE); 0.6 LightGBM + 0.4 CatBoost,
   each averaged over 3 fixed seeds.
5. **Validation** — expanding, out-of-time monthly folds (Jun–Sep), whole release cohorts.

Reproducible: `SEED = 2026` on every stochastic component.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEED = 2026
np.random.seed(SEED)
SEEDS = [SEED, SEED + 1, SEED + 2]
RUN_VALIDATION = True          # set False to skip the ~backtest and only build the submission

def find_data_dir():
    for root in [Path("."), Path("data"), Path("../input")] + list(Path("/kaggle/input").glob("*")):
        if (root / "train.csv").exists():
            return root
    raise FileNotFoundError("Put the 7 competition CSVs next to this notebook (or in ./data).")

DATA = find_data_dir()
OUT = Path(".")
print("data dir:", DATA.resolve())
print("versions: pandas", pd.__version__, "| lightgbm", lgb.__version__)

data dir: /projects/sandbox/nbtest
versions: pandas 2.3.3 | lightgbm 4.6.0


## 1. Load data

In [2]:
train_raw    = pd.read_csv(DATA / "train.csv",        parse_dates=["date_show"])
test_history = pd.read_csv(DATA / "test_history.csv", parse_dates=["date_show"])
test_raw     = pd.read_csv(DATA / "test.csv",         parse_dates=["date_show"])
sample       = pd.read_csv(DATA / "sample_submission.csv")
for name, df in [("train", train_raw), ("test_history", test_history), ("test", test_raw)]:
    print(f"{name:13s} {df.shape}  {df.date_show.min().date()} -> {df.date_show.max().date()}")

train         (138959, 7)  2025-04-01 -> 2025-09-30
test_history  (32323, 7)  2025-10-01 -> 2026-03-20
test          (72611, 5)  2025-10-04 -> 2026-03-27


## 2. Auxiliary tables (holidays, movie metadata, ticket prices)

In [3]:
def norm_title(s):
    return (s.astype(str).str.upper().str.strip()
            .str.replace(r"\s*\((?:3D|IMAX[^)]*)\)\s*$", "", regex=True).str.strip())

def load_aux():
    hol = pd.read_csv(DATA / "holidays.csv", parse_dates=["date"]).rename(columns={"date": "target_date"})
    hol["is_holiday"] = hol["holiday_tipe"].eq("holiday").astype("int8")
    hol = hol[["target_date", "is_holiday", "day_tipe"]]

    movies = pd.read_csv(DATA / "movies.csv")
    movies["base_title"] = norm_title(movies["original_title"])
    movies["genre_primary"] = movies["genre"].fillna("UNK").str.split(",").str[0].str.strip()
    movies = movies[["base_title", "age_rating", "genre_primary"]].drop_duplicates("base_title")

    prices = pd.read_csv(DATA / "ticket_prices.csv")
    prices["price_kind"] = prices["price_day"].str.lower()
    prices = prices.pivot_table(index="city_name", columns="price_kind", values="ceil", aggfunc="median")
    prices.columns = [f"price_{x}" for x in prices.columns]
    return hol, movies, prices.reset_index()

HOLIDAYS, MOVIES, PRICES = load_aux()

## 3. Wide-release alignment (skip 1-cinema previews)

In [4]:
def wide_release_dates(raw, frac=0.3):
    raw = raw.copy()
    raw["movie_title"] = raw.movie_title.str.upper().str.strip()
    nc = raw.groupby(["movie_title", "date_show"]).cinema_ids.nunique().rename("n").reset_index()
    first = nc.groupby("movie_title").date_show.transform("min")
    nc = nc[nc.date_show <= first + pd.Timedelta(days=21)]
    peak = nc.groupby("movie_title").n.transform("max")
    return nc[nc.n >= frac * peak].groupby("movie_title").date_show.min()

def align_to_wide_release(raw, frac=0.3):
    raw = raw.copy()
    raw["movie_title"] = raw.movie_title.str.upper().str.strip()
    first_seen = raw.groupby("movie_title").date_show.min()
    d1 = wide_release_dates(raw, frac)
    out = raw[raw.date_show >= raw.movie_title.map(d1)]
    # titles already running when the data starts are left-censored: kept as
    # competition context but never used as a training cohort
    censored = first_seen[first_seen == raw.date_show.min()].index
    return out, set(censored)

## 4. Per-pair D1–D3 history features

In [5]:
HIST_COLS = ["total_ticket", "occupation_rate", "total_show"]

def pair_history_features(panel_hist, pairs, movie, d1, hist_cols=HIST_COLS):
    """panel_hist: (cinema, date) x hist_cols for the 3 calendar days D1-D3 (missing = 0)."""
    hist = panel_hist.reset_index()
    hist["day"] = (hist.date_show - d1).dt.days + 1
    wide = hist.pivot(index="cinema_ids", columns="day", values=hist_cols)
    wide.columns = [f"{name}_d{day}" for name, day in wide.columns]
    wide = wide.reset_index().merge(pairs, on="cinema_ids", how="left")
    for day in (1, 2, 3):
        for col in hist_cols:
            if f"{col}_d{day}" not in wide:
                wide[f"{col}_d{day}"] = 0.0
    wide["scale"] = wide[[f"total_ticket_d{x}" for x in (1, 2, 3)]].sum(axis=1).div(3).clip(lower=1)
    for day in (1, 2, 3):
        wide[f"r_d{day}"] = wide[f"total_ticket_d{day}"] / wide.scale
        wide[f"present_d{day}"] = wide[f"total_ticket_d{day}"].gt(0).astype("int8")
    wide["r_slope"] = (wide.r_d3 - wide.r_d1) / 2
    wide["show_slope"] = (wide.total_show_d3 - wide.total_show_d1) / 2
    wide["occ_slope"] = (wide.occupation_rate_d3 - wide.occupation_rate_d1) / 2
    wide["tickets_per_show_d3"] = wide.total_ticket_d3 / wide.total_show_d3.clip(lower=1)
    wide["movie_pair_count"] = len(wide)
    wide["movie_total_d1"] = wide.total_ticket_d1.sum()
    wide["movie_total_d2"] = wide.total_ticket_d2.sum()
    wide["movie_total_d3"] = wide.total_ticket_d3.sum()
    wide["movie_active_d1"] = int(wide.present_d1.sum())
    wide["movie_active_d2"] = int(wide.present_d2.sum())
    wide["pair_scale_pct"] = wide.scale.rank(pct=True)
    wide["pair_scale_share"] = wide.scale / max(float(wide.scale.sum()), 1.0)
    wide["movie_r_slope"] = (wide.movie_total_d3 - wide.movie_total_d1) / max(float(wide.movie_total_d1.iloc[0]), 1.0)
    wide["movie"] = movie
    wide["base_title"] = norm_title(pd.Series([movie])).iloc[0]
    wide["release_date"] = d1
    wide["release_dow"] = d1.dayofweek
    return wide


def build_dense_cohorts(raw):
    """Train: for each title, pairs active on calendar D3 -> all 7 days D4-D10 (missing rows = 0)."""
    raw = raw.copy()
    raw["movie_title"] = raw.movie_title.str.upper().str.strip()
    data_min, data_max = raw.date_show.min(), raw.date_show.max()
    out = []
    for movie, gm in raw.groupby("movie_title", sort=False):
        d1 = gm.date_show.min()
        d3, d10 = d1 + pd.Timedelta(days=2), d1 + pd.Timedelta(days=9)
        if d1 == data_min or d10 > data_max:           # left-censored or incomplete targets
            continue
        d3_rows = gm[gm.date_show.eq(d3)]
        if d3_rows.empty:
            continue
        pairs = d3_rows[["cinema_ids", "city_name"]].drop_duplicates("cinema_ids")
        idx = pd.MultiIndex.from_product([pairs.cinema_ids.tolist(), pd.date_range(d1, d10)],
                                         names=["cinema_ids", "date_show"])
        panel = (gm.drop_duplicates(["cinema_ids", "date_show"], keep="last")
                   .set_index(["cinema_ids", "date_show"])[HIST_COLS].reindex(idx).fillna(0.0))
        wide = pair_history_features(panel.loc[(slice(None), pd.date_range(d1, d3)), :], pairs, movie, d1)

        future = panel.loc[(slice(None), pd.date_range(d3 + pd.Timedelta(days=1), d10)), "total_ticket"].reset_index()
        future["horizon"] = (future.date_show - d3).dt.days
        future = future.rename(columns={"date_show": "target_date", "total_ticket": "y"})
        cohort = future.merge(wide, on="cinema_ids", how="left")
        cohort["target_dow"] = cohort.target_date.dt.dayofweek
        cohort["ratio"] = cohort.y / cohort.scale
        out.append(cohort)
    return pd.concat(out, ignore_index=True)


def build_test_features(history, test):
    """Test: same features from test_history D1-D3 for every requested (id) row."""
    history, test = history.copy(), test.copy()
    for f in (history, test):
        f["movie_title"] = f.movie_title.str.upper().str.strip()
    out = []
    for movie, gt in test.groupby("movie_title", sort=False):
        gm = history[history.movie_title.eq(movie)]
        start = gt.date_show.min()
        d1, d3 = start - pd.Timedelta(days=3), start - pd.Timedelta(days=1)
        pairs = gt[["cinema_ids", "city_name"]].drop_duplicates("cinema_ids")
        idx = pd.MultiIndex.from_product([pairs.cinema_ids.tolist(), pd.date_range(d1, d3)],
                                         names=["cinema_ids", "date_show"])
        panel = (gm.drop_duplicates(["cinema_ids", "date_show"], keep="last")
                   .set_index(["cinema_ids", "date_show"])[HIST_COLS].reindex(idx).fillna(0.0))
        wide = pair_history_features(panel, pairs, movie, d1)
        cohort = gt.rename(columns={"date_show": "target_date"}).merge(wide, on=["cinema_ids", "city_name"], how="left")
        cohort["horizon"] = (cohort.target_date - d3).dt.days
        cohort["target_dow"] = cohort.target_date.dt.dayofweek
        out.append(cohort)
    return pd.concat(out, ignore_index=True)

## 5. Calendar, holiday, price and genre features

In [6]:
def add_aux(df):
    x = df.merge(HOLIDAYS, on="target_date", how="left")
    x = x.merge(MOVIES, on="base_title", how="left")
    x = x.merge(PRICES, on="city_name", how="left")
    x["is_holiday"] = x.is_holiday.fillna(0).astype("int8")
    x["day_tipe"] = x.day_tipe.fillna("unknown")
    x["age_rating"] = x.age_rating.fillna("UNK")
    x["genre_primary"] = x.genre_primary.fillna("UNK")
    kind = np.where(x.target_dow.eq(4), "friday", np.where(x.target_dow.ge(5), "weekend", "weekday"))
    x["price_target"] = [x.iloc[i].get(f"price_{k}", np.nan) for i, k in enumerate(kind)]
    return x

## 6. Screen-competition features

Built only from each title's own D1–D3 window — exactly what `test_history.csv` exposes — so train and test
features are constructed identically and no future labels are used.

In [7]:
def history_windows(raw):
    raw = raw.copy()
    raw["movie_title"] = raw.movie_title.str.upper().str.strip()
    raw["d1"] = raw.movie_title.map(raw.groupby("movie_title").date_show.min())
    raw["day"] = (raw.date_show - raw.d1).dt.days + 1
    return raw[raw.day.between(1, 3)]


def add_competition(df, hist):
    df = df.copy()
    h = hist.rename(columns={"date_show": "target_date"})

    # other titles in their D1-D3 window at the same cinema on the target date
    cell = (h.groupby(["cinema_ids", "target_date", "movie_title"])
              .agg(shows=("total_show", "sum"), tix=("total_ticket", "sum"),
                   opening=("day", lambda s: int((s == 1).any())))
              .reset_index())
    key = df[["cinema_ids", "target_date", "movie"]].drop_duplicates()
    j = key.merge(cell, on=["cinema_ids", "target_date"], how="left")
    j = j[j.movie_title.notna() & (j.movie_title != j.movie)]
    agg = (j.groupby(["cinema_ids", "target_date", "movie"])
             .agg(comp_titles=("movie_title", "nunique"), comp_shows=("shows", "sum"),
                  comp_tix=("tix", "sum"), comp_openings=("opening", "sum")).reset_index())
    df = df.merge(agg, on=["cinema_ids", "target_date", "movie"], how="left")

    # cumulative new openings at this cinema between D3 and the target date
    opens = (h[h.day == 1][["cinema_ids", "target_date", "movie_title", "total_show"]]
               .rename(columns={"target_date": "open_date", "total_show": "open_shows"}))
    k2 = df[["cinema_ids", "movie", "release_date", "target_date"]].drop_duplicates().merge(opens, on="cinema_ids", how="left")
    k2 = k2[(k2.movie_title != k2.movie) & (k2.open_date > k2.release_date + pd.Timedelta(days=2))
            & (k2.open_date <= k2.target_date)]
    cum = (k2.groupby(["cinema_ids", "movie", "target_date"])
             .agg(cum_new_titles=("movie_title", "nunique"), cum_new_shows=("open_shows", "sum")).reset_index())
    df = df.merge(cum, on=["cinema_ids", "movie", "target_date"], how="left")

    # national: titles opening between D3 and the target date, and their D1 strength
    nat = (h[h.day == 1].groupby(["movie_title", "target_date"])
             .agg(n_cin=("cinema_ids", "nunique"), d1_tix=("total_ticket", "sum"))
             .reset_index().rename(columns={"target_date": "open_date"}))
    k3 = df[["movie", "release_date", "target_date"]].drop_duplicates()
    k3 = k3.assign(_k=1).merge(nat.assign(_k=1), on="_k").drop(columns="_k")
    k3 = k3[(k3.movie_title != k3.movie) & (k3.open_date > k3.release_date + pd.Timedelta(days=2))
            & (k3.open_date <= k3.target_date)]
    nagg = (k3.groupby(["movie", "target_date"])
              .agg(nat_new_titles=("movie_title", "nunique"), nat_new_cinemas=("n_cin", "sum"),
                   nat_new_d1tix=("d1_tix", "sum")).reset_index())
    df = df.merge(nagg, on=["movie", "target_date"], how="left")

    # this film's share of visible screenings at the cinema on D3
    d3cell = (h.groupby(["cinema_ids", "target_date"]).total_show.sum().rename("cinema_visible_shows_d3")
                .reset_index().rename(columns={"target_date": "d3_date"}))
    df["d3_date"] = df.release_date + pd.Timedelta(days=2)
    df = df.merge(d3cell, on=["cinema_ids", "d3_date"], how="left")
    df["own_show_share_d3"] = df.total_show_d3 / df.cinema_visible_shows_d3.clip(lower=1)

    fill0 = ["comp_titles", "comp_shows", "comp_tix", "comp_openings", "cum_new_titles",
             "cum_new_shows", "nat_new_titles", "nat_new_cinemas", "nat_new_d1tix"]
    df[fill0] = df[fill0].fillna(0)
    df["comp_shows_vs_own"] = df.comp_shows / df.total_show_d3.clip(lower=1)
    df["cum_new_shows_vs_own"] = df.cum_new_shows / df.total_show_d3.clip(lower=1)
    df["nat_new_d1tix_vs_movie"] = df.nat_new_d1tix / df.movie_total_d1.clip(lower=1)
    df["movie_mean_scale"] = df.groupby("movie").scale.transform("mean")
    df["movie_mean_occ_d3"] = df.groupby("movie").occupation_rate_d3.transform("mean")
    df["movie_d3_d1"] = df.movie_total_d3 / df.movie_total_d1.clip(lower=1)
    df["is_weekend"] = df.target_dow.ge(5).astype(int)
    df["week2"] = (df.target_date >= df.release_date + pd.Timedelta(days=7)).astype(int)
    return df


NUM_FEATURES = [
    "scale", "r_d1", "r_d2", "r_d3", "r_slope", "present_d1", "present_d2", "present_d3",
    "occupation_rate_d1", "occupation_rate_d2", "occupation_rate_d3", "occ_slope",
    "total_show_d1", "total_show_d2", "total_show_d3", "show_slope", "tickets_per_show_d3",
    "movie_pair_count", "movie_total_d1", "movie_total_d2", "movie_total_d3",
    "movie_active_d1", "movie_active_d2", "pair_scale_pct", "pair_scale_share", "movie_r_slope",
    "horizon", "target_dow", "release_dow", "is_holiday", "price_target",
]
COMP_FEATURES = [
    "comp_titles", "comp_shows", "comp_tix", "comp_openings", "cum_new_titles", "cum_new_shows",
    "nat_new_titles", "nat_new_cinemas", "nat_new_d1tix", "own_show_share_d3", "cinema_visible_shows_d3",
    "comp_shows_vs_own", "cum_new_shows_vs_own", "nat_new_d1tix_vs_movie", "movie_mean_scale",
    "movie_mean_occ_d3", "movie_d3_d1", "is_weekend", "week2",
]
CAT_FEATURES = ["cinema_ids", "city_name", "genre_primary", "age_rating", "day_tipe"]
FEATURES = NUM_FEATURES + COMP_FEATURES + CAT_FEATURES
print(len(FEATURES), "features")

55 features


## 7. Build train cohorts and test features

In [8]:
aligned, censored = align_to_wide_release(train_raw)
train = add_aux(build_dense_cohorts(aligned))
train = train[~train.movie.isin(censored)]
train = add_competition(train, history_windows(aligned))

test = add_competition(add_aux(build_test_features(test_history, test_raw)), history_windows(test_history))

print(f"train: {len(train):,} rows | {train.movie.nunique()} movies | zero-target rate {train.y.eq(0).mean():.3f}")
print(f"test : {len(test):,} rows | {test.movie.nunique()} movies | unique ids {test.id.nunique():,}")

train: 51,926 rows | 171 movies | zero-target rate 0.326
test : 72,611 rows | 160 movies | unique ids 72,611


## 8. Model: LightGBM + CatBoost on `ticket / scale` (L1 loss == MASE)

In [9]:
LGB_PARAMS = dict(objective="regression_l1", n_estimators=300, learning_rate=0.03, num_leaves=15,
                  min_child_samples=80, reg_lambda=20, colsample_bytree=0.7, subsample=0.8,
                  subsample_freq=1, verbosity=-1, n_jobs=-1)
CAT_PARAMS = dict(loss_function="MAE", iterations=600, depth=6, learning_rate=0.05, l2_leaf_reg=10,
                  verbose=False, thread_count=-1, allow_writing_files=False)
CAT_WEIGHT = 0.4


def lgb_frames(tr, va):
    a, b = tr[FEATURES].copy(), va[FEATURES].copy()
    for c in CAT_FEATURES:
        cats = pd.Index(pd.concat([a[c], b[c]]).astype(str).unique())
        a[c] = pd.Categorical(a[c].astype(str), categories=cats)
        b[c] = pd.Categorical(b[c].astype(str), categories=cats)
    return a, b


def cat_frames(tr, va):
    a, b = tr[FEATURES].copy(), va[FEATURES].copy()
    for c in CAT_FEATURES:
        a[c] = a[c].fillna("UNK").astype(str)
        b[c] = b[c].fillna("UNK").astype(str)
    return a.fillna(-999), b.fillna(-999)


def fit_predict(tr, va):
    la, lb_ = lgb_frames(tr, va)
    ca, cb = cat_frames(tr, va)
    lp, cp = np.zeros(len(va)), np.zeros(len(va))
    for s in SEEDS:
        lp += lgb.LGBMRegressor(**LGB_PARAMS, random_state=s).fit(
            la, tr.ratio, categorical_feature=CAT_FEATURES).predict(lb_)
        cp += CatBoostRegressor(**CAT_PARAMS, random_seed=s).fit(
            ca, tr.ratio, cat_features=CAT_FEATURES).predict(cb)
    lp, cp = np.clip(lp / len(SEEDS), 0, None), np.clip(cp / len(SEEDS), 0, None)
    return (1 - CAT_WEIGHT) * lp + CAT_WEIGHT * cp

## 9. Out-of-time validation (expanding monthly folds)

In [10]:
def validate(d, months=("2025-06", "2025-07", "2025-08", "2025-09")):
    rows = []
    for m in months:
        start = pd.Timestamp(m + "-01"); end = start + pd.offsets.MonthBegin(1)
        tr = d[d.release_date + pd.Timedelta(days=9) < start]          # only cohorts fully finished before
        va = d[(d.release_date >= start) & (d.release_date < end)]
        p = fit_predict(tr, va)
        decay = va.horizon.map(tr.groupby("horizon").ratio.median()).to_numpy()
        rows.append(dict(fold=m, v6=np.abs(va.ratio - p).mean(),
                         decay_baseline=np.abs(va.ratio - decay).mean(),
                         flat_baseline=np.abs(va.ratio - 1).mean(),
                         movies=va.movie.nunique(), rows=len(va), zero_rate=va.y.eq(0).mean()))
    rep = pd.DataFrame(rows)
    print(rep.round(4).to_string(index=False))
    print("mean:", rep[["v6", "decay_baseline", "flat_baseline"]].mean().round(4).to_dict())
    return rep

if RUN_VALIDATION:
    cv_report = validate(train)

   fold     v6  decay_baseline  flat_baseline  movies  rows  zero_rate
2025-06 0.4199          0.4126         0.6634      27  7469     0.4090
2025-07 0.3810          0.4756         0.7239      35 10801     0.2641
2025-08 0.3839          0.4428         0.7554      34 10479     0.3625
2025-09 0.4564          0.4836         0.7694      22  7518     0.2809
mean: {'v6': 0.4103, 'decay_baseline': 0.4536, 'flat_baseline': 0.728}


## 10. Fit on all train cohorts and write `submission.csv`

In [11]:
ratio = fit_predict(train, test)
tickets = np.rint(ratio * test.scale.to_numpy()).clip(0).astype(int)
pred = pd.DataFrame({"id": test.id.astype(int), "total_ticket": tickets})

submission = sample[["id"]].merge(pred, on="id", how="left", validate="one_to_one")
assert submission.total_ticket.notna().all(), "missing predictions"
submission["total_ticket"] = submission.total_ticket.astype(int)

assert list(submission.columns) == ["id", "total_ticket"]
assert len(submission) == len(sample) and submission.id.equals(sample.id) and submission.id.is_unique
assert (submission.total_ticket >= 0).all()

submission.to_csv(OUT / "submission.csv", index=False)
print("saved", (OUT / "submission.csv").resolve(), submission.shape)
print(submission.total_ticket.describe().round(1).to_dict())
submission.head()

saved /projects/sandbox/nbtest/submission.csv (72611, 2)
{'count': 72611.0, 'mean': 153.5, 'std': 627.6, 'min': 0.0, '25%': 5.0, '50%': 32.0, '75%': 119.0, 'max': 36328.0}


,id,total_ticket
0,1,22
1,2,19
2,3,18
3,4,0
4,5,0


In [12]:
import hashlib
print("md5:", hashlib.md5((OUT / "submission.csv").read_bytes()).hexdigest())
# the LB 0.4777 file has md5 5a0ea01edd6f946cd330110fff5a4c98 (same library versions)

md5: 5a0ea01edd6f946cd330110fff5a4c98
